## Lernziel

Mehrere PEExcel-Dateien über die vorhandene Importschnittstelle zusammenführen und ihren Stundenbestand prüfen.

**Voraussetzungen:** Gespeicherte PEExcel-Exporte; keine Excel-Installation erforderlich.

Einmalig im Repository installieren: `python -m pip install -e ".[publication]"`.
Zum eigenen Ausführen das Notebook im geklonten Repository öffnen und die Zellen von oben nach unten ausführen.
Die Website zeigt gespeicherte Ausgaben; sie führt keinen Python-Code im Browser aus.

[Tutorial-Notebook herunterladen](downloads/tutorial_zeitreihen.ipynb) · [Alle Tutorials](python.qmd)


## 1. Quelldateien auswählen

Das Beispiel startet mit dem versionierten F7-Export. Ergänze `DATEIEN` um weitere Exporte mit eindeutigen Szenarionamen.
Es werden keine Dateien aus persönlichen Ordnern automatisch eingesammelt.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Im geklonten Repository ausführen, z. B. aus website/ oder notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pexl").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Bitte das Notebook innerhalb des peexcel-Repositorys öffnen.")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
OUTPUT = ROOT / "notebooks/local_exports/tutorials"
OUTPUT.mkdir(parents=True, exist_ok=True)
from pexl import Project
DATEIEN = [ROOT / "data/exports/Aspern PVCheck _ F7.xlsx"]
assert DATEIEN and all(p.is_file() for p in DATEIEN)
projekt = Project.from_files(DATEIEN)
print("Importwarnungen:", projekt.warnings)

Importwarnungen: []


## 2. Vollständigkeit prüfen

`Project.from_files` führt alle Szenarien zusammen und lehnt doppelte Schlüssel ab.
Jahresergebnisse allein genügen für Stundenanalysen nicht: jede ausgewählte Variante braucht Zeitreihen.

In [2]:
zeilen = []
for szenario in projekt:
    if szenario.timeseries is None:
        raise ValueError(f"Zeitreihe fehlt: {szenario.column_name}")
    frame = szenario.timeseries.to_frame()
    assert len(frame) == 8760, szenario.column_name
    zeilen.append({"Szenario": szenario.column_name, "Stunden": len(frame), "Kanäle": len(frame.columns)})
display(pd.DataFrame(zeilen))

,Szenario,Stunden,Kanäle
0,Aspern PVCheck _ F7,8760,15


## 3. Optional einen lokalen Zwischenstand speichern

Pickle erhält hier das Python-Projekt einschließlich Zeitreihen. Es ist kein langfristiges Austauschformat:
zum Wiederherstellen dieselbe Paketversion verwenden und nur selbst erzeugte, vertrauenswürdige Dateien laden.
Alternativ das Projekt jederzeit mit `Project.from_files(DATEIEN)` aus den Originalexporten neu aufbauen.

In [3]:
import pickle
zwischenspeicher = OUTPUT / "tutorial_zeitreihen.pkl"
with zwischenspeicher.open("wb") as f:
    pickle.dump(projekt, f, protocol=pickle.HIGHEST_PROTOCOL)
with zwischenspeicher.open("rb") as f:
    wiederhergestellt = pickle.load(f)
assert wiederhergestellt.column_names() == projekt.column_names()
for links, rechts in zip(projekt, wiederhergestellt):
    pd.testing.assert_frame_equal(links.timeseries.to_frame(), rechts.timeseries.to_frame())
print("Szenarien und Zeitreihen unverändert wiederhergestellt.")

Szenarien und Zeitreihen unverändert wiederhergestellt.


## Ergebnis

Das Projekt enthält die ausgewählten Varianten mit ihren Stundenwerten.
Das [Netzlast-Tutorial](tutorial_netzlast.ipynb) verwendet denselben Importweg ohne Abhängigkeit von einer Pickle-Datei.



## Weiterarbeiten

[Zur Tutorial-Übersicht](python.qmd). Die ursprünglichen Arbeitsnotebooks bleiben erhalten: [howto_create_project_from_different_timeseries_files](https://github.com/simonschaluppe/peexcel/blob/master/notebooks/howto_create_project_from_different_timeseries_files.ipynb).
